# NB 61 — Subspace λ scan: locating the optimal fluency dial

The subspace counterpart of the frame λ-precision sweep **NB 41**. Sweeps
**λ ∈ {0, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0}** for both subspace methods
(**pooled**, **hierarchical**) on the 10 FRH pairs (NB 33), blending each pair
into one `ConceptSubspace.blend` guide and steering through the same greedy
top-k loop (`k=4`, `STEPS=24`). λ=0 and λ=0.4 are **reused from NB 57**
(`57_subspace_ml.jsonl`); this notebook only generates the five new points.

The optimum is read two ways per method: the **unconstrained** argmax of joint
expression Δ, and the **guardrail-constrained** λ\* — the largest steering
subject to perplexity ratio ≤ 2.5 (the criterion that fixed the frame recipe at
λ≈0.4). Multilingual (8 languages, `MIN_LEMMAS=11`), matching the reused points.

*(Subspace has no gauge-invariant angle logging, so the knee uses expression Δ,
both-improved rate, and the ppl guardrail — not token-cosine as in NB 41.)*

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.nlp import SupportedLanguages
from subspace import ConceptSubspace, SubspaceUnembeddingRepresentation

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS, K, STEPS = 3, 4, 24
RANK, R_LEMMA = 8, 1
ARMS = ["pooled", "hierarchical"]
LAMBDAS_NEW = [0.3, 0.5, 0.6, 0.8, 1.0]  # λ=0, 0.4 reused from NB 57
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"

REUSE_JSONL = "resources/57_subspace_ml.jsonl"  # blend records at λ=0, 0.4
PAIRS = json.loads(Path("resources/33_scaling_groups_bf16.json").read_text())["pairs"]

SMOKE = True  # subset + `_smoke` outputs; flip to False for the full sweep


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

if SMOKE:
    PAIRS, QUESTIONS, LAMBDAS_NEW = PAIRS[:1], QUESTIONS[:2], [0.5]

SUFFIX = "_smoke" if SMOKE else ""
LOG_PATH = Path(f"resources/61_subspace_lambda_scan{SUFFIX}.jsonl")
LOG_PATH.unlink(missing_ok=True)
print(f"{len(PAIRS)} pairs x {len(ARMS)} arms x {len(LAMBDAS_NEW)} new lambda x {len(QUESTIONS)} prompts")

In [ ]:
sur = SubspaceUnembeddingRepresentation.from_model_id(
    MODEL_ID,
    rank=RANK,
    r_lemma=R_LEMMA,
    language_codes=SupportedLanguages.Llama,  # 8-language FRH set (matches NB 57)
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
prompts = [chat(u, a) for u, a in QUESTIONS]
baseline_texts = sur.generate(prompts, max_new_tokens=STEPS + 2, do_sample=False)

for arm in ARMS:
    harness = EvaluationHarness(sur, LOG_PATH, MIN_LEMMAS, MAX_TOKENS)
    harness.member_lemmas(PAIRS[0]["a"])  # force WordNet load out of the loop
    built = {}

    def concept(name):
        if name not in built:
            built[name] = sur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS, method=arm)
        return built[name]

    for grp in PAIRS:
        a, b = grp["a"], grp["b"]
        cs = {a: concept(a), b: concept(b)}
        guide = ConceptSubspace.blend(list(cs.values()))
        for lam in LAMBDAS_NEW:
            torch.cuda.empty_cache()
            texts, probe = sur.generate_with_topk_guide(
                prompts, guide=guide, k=K, steps=STEPS, fluency_weight=lam
            )
            harness.evaluate(
                prompts, texts, [a, b],
                config={
                    "mode": "blend", "model": MODEL_KEY, "arm": arm, "lam": lam,
                    "a": a, "b": b, "pair_rho": grp.get("pair_rho"),
                },
                probe=probe, baseline_texts=baseline_texts, concepts=cs, angles=False,
            )
    print(f"{arm}: done")

## λ-response: combine the new points with the reused λ=0, 0.4

In [ ]:
def blend_rows(path, lams=None):
    rows = []
    for line in Path(path).read_text().strip().split("\n"):
        if not line:
            continue
        r = json.loads(line)
        c = r["config"]
        if c.get("mode") != "blend":
            continue
        if lams is not None and c["lam"] not in lams:
            continue
        ds = [r["expression"][c["a"]]["delta"], r["expression"][c["b"]]["delta"]]
        rows.append({
            "arm": c["arm"], "lam": c["lam"],
            "joint_delta": min(ds), "both_improved": all(x > 0 for x in ds),
            "ppl_ratio": r["ppl_ratio"],
        })
    return pd.DataFrame(rows)


results = pd.concat(
    [blend_rows(LOG_PATH), blend_rows(REUSE_JSONL, lams={0.0, 0.4})],
    ignore_index=True,
)
summary = (
    results.groupby(["arm", "lam"])
    .agg(
        joint_delta=("joint_delta", "mean"),
        both_improved=("both_improved", "mean"),
        ppl_ratio=("ppl_ratio", "mean"),
        n=("joint_delta", "size"),
    )
    .reset_index()
    .sort_values(["arm", "lam"])
)
summary.to_csv(f"resources/61_subspace_lambda_scan_summary{SUFFIX}.csv", index=False)
print(summary.round(3).to_string(index=False))

print("\nknee per method (max joint Δ; and λ* s.t. ppl ≤ 2.5):")
for arm in ARMS:
    s = summary[summary.arm == arm].sort_values("lam")
    uncon = s.loc[s.joint_delta.idxmax()]
    ok = s[s.ppl_ratio <= 2.5]
    if len(ok):
        con = ok.loc[ok.joint_delta.idxmax()]
        con_str = f"λ*={con.lam} (joint Δ={con.joint_delta:.3f}, ppl={con.ppl_ratio:.2f})"
    else:
        m = s.loc[s.ppl_ratio.idxmin()]
        con_str = f"none ≤ 2.5; min ppl={m.ppl_ratio:.2f} at λ={m.lam}"
    print(f"  {arm:12s}: unconstrained λ={uncon.lam} (joint Δ={uncon.joint_delta:.3f}, "
          f"ppl={uncon.ppl_ratio:.2f}) | guardrail {con_str}")

In [ ]:
arm_colors = {"pooled": "tab:gray", "hierarchical": "tab:blue"}
panels = [
    ("joint_delta", "mean joint Δ (min over pair)"),
    ("both_improved", "both-improved rate"),
    ("ppl_ratio", "mean perplexity ratio"),
]
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4.4))
for ax, (metric, label) in zip(axes, panels):
    for arm in ARMS:
        s = summary[summary.arm == arm].sort_values("lam")
        ax.plot(s["lam"], s[metric], marker="o", color=arm_colors[arm], label=arm)
    ax.set_xlabel("fluency weight λ")
    ax.set_ylabel(label)
axes[0].axhline(0, color="gray", lw=0.6)
axes[2].axhline(2.5, color="red", ls=":", lw=0.8)  # fluency guardrail
axes[0].legend()
plt.suptitle("Subspace blend λ-response (n=2 pairs): pooled vs hierarchical")
plt.tight_layout()
plt.savefig(f"resources/61_subspace_lambda_scan{SUFFIX}.png", dpi=150)
plt.show()